<a href="https://colab.research.google.com/github/alessiobsc/AI4Cyber-FaceRecognitionSecurity/blob/main/BIM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drive Mount and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

In [ ]:
!pip install adversarial-robustness-toolbox # ART libreria per generare e valutare attacchi adversarial
!pip install torch torchvision
!pip install facenet_pytorch

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

# Import da ART
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.defences.preprocessor import SpatialSmoothing

# Utility import
utils_path = "/content/drive/MyDrive/Gruppo IA4Cyber"
if utils_path not in sys.path: sys.path.append(utils_path)

from cyber_utils import save_npz_attack

# Verifica che la GPU sia disponibile
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo in uso: {device}")

# Prepare test data

In [ ]:
from facenet_pytorch import InceptionResnetV1
#Carica un modello già allenato sul dataset VGGFace2 e lo mette in fase di inferenza
resnet = InceptionResnetV1(pretrained='vggface2').eval()
#modalità classificazione diretta
resnet.classify = True

resnet.to(device)

In [ ]:
# Incapsula la rete PyTorch in un classificatore ART.
# Questo wrapper permette ad ART di calcolare gradienti, predizioni
# e attacchi adversarial sul modello NN1. Sono specificati loss,
# ottimizzatore, range dei valori di input, shape delle immagini
# e numero totale di classi VGGFace2.

import torch
from art.estimators.classification import PyTorchClassifier
import torch.optim as optim

# Definizione componenti per ART
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.01)

# Creazione del wrapper con device esplicito
classifier = PyTorchClassifier(
    model=resnet,
    clip_values=(-1.0, 1.0),
    loss=criterion,
    optimizer=optimizer,
    input_shape=(3, 160, 160),
    nb_classes=8631,
    device_type=device.type # Passa 'cuda' o 'cpu' dinamicamente
)

In [ ]:
import tensorflow as tf
import pandas as pd

# Caricamento delle etichette (VGGFace2) del classificatore
fpath = tf.keras.utils.get_file('rcmalli_vggface_labels_v2.npy',
                             "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
                             cache_subdir="./")
LABELS = np.load(fpath)

path_drive = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto/train'

csv_meta_path = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/identity_meta.csv'

try:
    meta_df = pd.read_csv(csv_meta_path, skipinitialspace=True, on_bad_lines='skip')
    meta_df.columns = meta_df.columns.str.strip()
    meta_df['Class_ID'] = meta_df['Class_ID'].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

    # Crea il dizionario che mappa, ad esempio, 'n000156' -> ' Ajda_Pekkan'
    id_to_name = dict(zip(meta_df['Class_ID'], meta_df['Name']))
    print(f"Metadati caricati con successo.")
except Exception as e:
    print(f"Errore caricamento CSV metadati: {e}")
    id_to_name = {}

In [ ]:
# Prepara il test set in formato compatibile con ART.
# Ogni immagine viene ridimensionata, convertita in tensore e normalizzata
# nel range [-1, 1]. Le etichette testuali vengono convertite negli indici
# numerici usati dal classificatore VGGFace2.

import numpy as np

# Mapping inverso: Nome Persona -> Indice numerico del modello
name_to_idx = {name.strip().replace("'", ""): i for i, name in enumerate(LABELS)}

x_test_list = []
y_test_list = []

print("Caricamento delle 1000 immagini del test set...")

for folder_name in os.listdir(path_drive):
    folder_path = os.path.join(path_drive, folder_name)
    if not os.path.isdir(folder_path): continue

    real_name = id_to_name.get(folder_name, "").strip().replace("'", "")
    label_idx = name_to_idx.get(real_name, -1)
    if label_idx == -1: continue

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        try:
            # Preprocessing identico alla baseline (78.5%)
            img_pil = Image.open(img_path).convert('RGB').resize((160, 160))
            img_tns = transforms.ToTensor()(img_pil)
            # Normalizzazione [-1, 1] come deciso
            img_tns = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])(img_tns)

            x_test_list.append(img_tns.numpy())
            y_test_list.append(label_idx)
        except:
            continue

x_test_all = np.array(x_test_list)
y_test_all = np.array(y_test_list)

print(f"Completato! Dataset pronto: {x_test_all.shape}")

# Initialize the attack

In [ ]:
# Genera adversarial examples con FGSM targeted verso Alessio Cerci.
# Per ogni valore di epsilon, misura l'Attack Success Rate,
# cioè la percentuale di immagini che il modello classifica come Cerci.
# Calcola inoltre la distanza L2 media per valutare l'invasività
# della perturbazione.

# Cerchiamo Alessio Cerci nell'array LABELS usato dal modello
target_name_ricerca = "Alessio_Cerci"
clean_labels = [str(l).strip().replace("'", "").replace('"', '') for l in LABELS]

try:
    real_target_idx = clean_labels.index(target_name_ricerca)
    print(f"L'indice CORRETTO per il modello è: {real_target_idx}")
    print(f"Nome confermato dal modello: {LABELS[real_target_idx]}")
except ValueError:
    # Se non lo trova, cerchiamo chi c'è vicino o con nome simile
    possibili = [i for i, s in enumerate(clean_labels) if "Cerci" in s]
    print(f"Indici che contengono 'Cerci': {possibili}")
    real_target_idx = possibili[0] if possibili else 250 # Fallback

BIM untargeted

In [ ]:
from art.attacks.evasion import BasicIterativeMethod
import numpy as np

epsilons = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
max_iter = 20

results_bim_untargeted = []

for epsilon in epsilons:

    print(f"\nBIM UNTARGETED | eps={epsilon}, max_iter={max_iter}")

    if epsilon == 0.0:
        x_adv_bim_untargeted = x_test_all.copy()
        epsilon_step = 0.0
    else:
        epsilon_step = epsilon / max_iter

        attack_bim_untargeted = BasicIterativeMethod(
            estimator=classifier,
            eps=epsilon,
            eps_step=epsilon_step,
            max_iter=max_iter,
            targeted=False
        )

        x_adv_bim_untargeted = attack_bim_untargeted.generate(
            x=x_test_all,
            batch_size=128
        )

    preds_bim_untargeted = classifier.predict(
        x_adv_bim_untargeted,
        batch_size=128
    )

    pred_labels_bim_untargeted = np.argmax(preds_bim_untargeted, axis=1)

    accuracy_bim_untargeted = np.mean(pred_labels_bim_untargeted == y_test_all)
    perturbation_bim_untargeted = np.mean(np.abs(x_adv_bim_untargeted - x_test_all))

    print("eps_step: {:.5f}".format(epsilon_step))
    print("Accuracy: {:.2f}%".format(accuracy_bim_untargeted * 100))
    print("Average perturbation: {:.4f}".format(perturbation_bim_untargeted))

    results_bim_untargeted.append({
        "epsilon": epsilon,
        "epsilon_step": epsilon_step,
        "max_iter": max_iter,
        "accuracy": accuracy_bim_untargeted,
        "perturbation": perturbation_bim_untargeted
    })

BIM targeted

In [ ]:
from art.attacks.evasion import BasicIterativeMethod
import numpy as np

epsilons = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
max_iter = 20

target_name = "Alessio_Cerci"

clean_labels = [str(label).strip().replace("'", "").replace('"', '') for label in LABELS]
target_class = clean_labels.index(target_name)

print("Target class:", target_class)
print("Target name:", LABELS[target_class])

targeted_labels = np.ones(y_test_all.shape, dtype=np.int64) * target_class
one_hot_targeted_labels = np.eye(classifier.nb_classes)[targeted_labels]

results_bim_targeted = []

for epsilon in epsilons:

    print(f"\nBIM TARGETED | eps={epsilon}, max_iter={max_iter}")

    if epsilon == 0.0:
        x_adv_bim_targeted = x_test_all.copy()
        epsilon_step = 0.0
    else:
        epsilon_step = epsilon / max_iter

        attack_bim_targeted = BasicIterativeMethod(
            estimator=classifier,
            eps=epsilon,
            eps_step=epsilon_step,
            max_iter=max_iter,
            targeted=True
        )

        x_adv_bim_targeted = attack_bim_targeted.generate(
            x=x_test_all,
            y=one_hot_targeted_labels,
            batch_size=128
        )

    preds_bim_targeted = classifier.predict(
        x_adv_bim_targeted,
        batch_size=128
    )

    pred_labels_bim_targeted = np.argmax(preds_bim_targeted, axis=1)

    accuracy_original_labels = np.mean(pred_labels_bim_targeted == y_test_all)
    targeted_success_rate = np.mean(pred_labels_bim_targeted == target_class)
    perturbation_bim_targeted = np.mean(np.abs(x_adv_bim_targeted - x_test_all))

    print("eps_step: {:.5f}".format(epsilon_step))
    print("Accuracy rispetto alle label originali: {:.2f}%".format(accuracy_original_labels * 100))
    print("Targeted attack success rate: {:.2f}%".format(targeted_success_rate * 100))
    print("Average perturbation: {:.4f}".format(perturbation_bim_targeted))

    results_bim_targeted.append({
        "epsilon": epsilon,
        "epsilon_step": epsilon_step,
        "max_iter": max_iter,
        "accuracy_original_labels": accuracy_original_labels,
        "targeted_success_rate": targeted_success_rate,
        "perturbation": perturbation_bim_targeted
    })

## Save samples

In [ ]:
from art.attacks.evasion import FastGradientMethod
from cyber_utils import save_npz_attack_safe

In [ ]:
# === FASE 2: Attacco Finale BIM Untargeted e Salvataggio ===

BEST_EPSILON = 0.1
BEST_MAX_ITER = 20
BEST_EPSILON_STEP = BEST_EPSILON / BEST_MAX_ITER

print(
    f"Avvio attacco finale BIM untargeted: "
    f"eps={BEST_EPSILON}, eps_step={BEST_EPSILON_STEP}, max_iter={BEST_MAX_ITER}"
)

# 1. Ricreazione attacco definitivo
final_attack_bim_untargeted = BasicIterativeMethod(
    estimator=classifier,
    eps=BEST_EPSILON,
    eps_step=BEST_EPSILON_STEP,
    max_iter=BEST_MAX_ITER,
    targeted=False
)

x_adv_final_bim_untargeted = final_attack_bim_untargeted.generate(
    x=x_test_all,
    batch_size=128
)

# 2. Calcolo dei logit finali
print("Calcolo delle predizioni finali in corso...")
preds_final_bim_untargeted = classifier.predict(
    x_adv_final_bim_untargeted,
    batch_size=128
)

# 3. Salvataggio
save_npz_attack_safe(
    attack_family="BIM",
    attack_mode="untargeted_BEST",
    epsilon=BEST_EPSILON,
    x_adv=x_adv_final_bim_untargeted,
    preds_adv=preds_final_bim_untargeted,
    y_true=y_test_all,
    targeted=False
)

In [ ]:
# === FASE 2: Attacco Finale BIM Targeted e Salvataggio ===

BEST_EPSILON = 0.05
BEST_MAX_ITER = 20
BEST_EPSILON_STEP = BEST_EPSILON / BEST_MAX_ITER

target_name = "Alessio_Cerci"

clean_labels = [str(label).strip().replace("'", "").replace('"', '') for label in LABELS]
target_idx = clean_labels.index(target_name)

targeted_labels = np.ones(y_test_all.shape, dtype=np.int64) * target_idx
one_hot_targeted_labels = np.eye(classifier.nb_classes)[targeted_labels]

print(
    f"Avvio attacco finale BIM targeted verso {target_name}: "
    f"eps={BEST_EPSILON}, eps_step={BEST_EPSILON_STEP}, max_iter={BEST_MAX_ITER}"
)

# 1. Ricreazione attacco definitivo
final_attack_bim_targeted = BasicIterativeMethod(
    estimator=classifier,
    eps=BEST_EPSILON,
    eps_step=BEST_EPSILON_STEP,
    max_iter=BEST_MAX_ITER,
    targeted=True
)

x_adv_final_bim_targeted = final_attack_bim_targeted.generate(
    x=x_test_all,
    y=one_hot_targeted_labels,
    batch_size=128
)

# 2. Calcolo dei logit finali
print("Calcolo delle predizioni finali in corso...")
preds_final_bim_targeted = classifier.predict(
    x_adv_final_bim_targeted,
    batch_size=128
)

# 3. Salvataggio
save_npz_attack_safe(
    attack_family="BIM",
    attack_mode="targeted_BEST_Alessio_Cerci",
    epsilon=BEST_EPSILON,
    x_adv=x_adv_final_bim_targeted,
    preds_adv=preds_final_bim_targeted,
    y_true=y_test_all,
    targeted=True,
    target_idx=target_idx,
    target_name=target_name
)

Tabella risultati BIM


In [ ]:
import pandas as pd

df_bim_untargeted = pd.DataFrame(results_bim_untargeted)
df_bim_targeted = pd.DataFrame(results_bim_targeted)

print("BIM UNTARGETED")
display(df_bim_untargeted)

print("BIM TARGETED")
display(df_bim_targeted)

SEC BIM untargeted: Accuracy vs Epsilon

In [ ]:
import matplotlib.pyplot as plt

eps_unt = [r["epsilon"] for r in results_bim_untargeted]
acc_unt = [r["accuracy"] * 100 for r in results_bim_untargeted]

plt.figure(figsize=(8, 5))
plt.plot(eps_unt, acc_unt, marker="o")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy (%)")
plt.title("Security Evaluation Curve - BIM Untargeted")
plt.grid(True)
plt.show()

SEC BIM targeted: Success Rate vs Epsilon

In [ ]:
eps_tar = [r["epsilon"] for r in results_bim_targeted]
success_tar = [r["targeted_success_rate"] * 100 for r in results_bim_targeted]

plt.figure(figsize=(8, 5))
plt.plot(eps_tar, success_tar, marker="o")
plt.xlabel("Epsilon")
plt.ylabel("Targeted Attack Success Rate (%)")
plt.title("Security Evaluation Curve - BIM Targeted")
plt.grid(True)
plt.show()

Confronto BIM untargeted e targeted

In [ ]:
acc_tar_original = [r["accuracy_original_labels"] * 100 for r in results_bim_targeted]

plt.figure(figsize=(8, 5))
plt.plot(eps_unt, acc_unt, marker="o", label="BIM Untargeted Accuracy")
plt.plot(eps_tar, acc_tar_original, marker="o", label="BIM Targeted Accuracy on Original Labels")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy (%)")
plt.title("Security Evaluation Curves - BIM")
plt.legend()
plt.grid(True)
plt.show()

Perturbazione media BIM

In [ ]:
pert_unt = [r["perturbation"] for r in results_bim_untargeted]
pert_tar = [r["perturbation"] for r in results_bim_targeted]

plt.figure(figsize=(8, 5))
plt.plot(eps_unt, pert_unt, marker="o", label="BIM Untargeted")
plt.plot(eps_tar, pert_tar, marker="o", label="BIM Targeted")
plt.xlabel("Epsilon")
plt.ylabel("Average Perturbation")
plt.title("Average Perturbation - BIM")
plt.legend()
plt.grid(True)
plt.show()